In [4]:
import os
os.chdir(r'C:\Users\Klara\retail-intelligence')

import pandas as pd
import plotly.express as px

uncertainty = pd.read_csv('data/processed/uncertainty_metrics.csv')
inventory_risk = pd.read_csv('data/processed/inventory_risk.csv')

print(uncertainty[['StockCode', 'DataQualityFlag', 'MeanRelativeUncertainty']])

   StockCode                        DataQualityFlag  MeanRelativeUncertainty
0      23843  Low reliability - sparse/spike-driven                   1371.2
1      23166  Low reliability - sparse/spike-driven                   1056.7
2      21915                                     OK                    459.7
3      22086                                     OK                    431.6
4      22197                                     OK                    370.0
5      21977                                     OK                    345.5
6      15036                                     OK                    331.4
7      84077                                     OK                    317.6
8      84879                                     OK                    317.3
9      17003                                     OK                    314.6
10     23203                                     OK                    299.6
11     22469                                     OK                    294.7

In [5]:
fig = px.bar(
    uncertainty,
    x='StockCode',
    y='MeanRelativeUncertainty',
    color='DataQualityFlag',
    color_discrete_map={
        'OK': '#1f77b4',
        'Low reliability - sparse/spike-driven': '#d62728'
    },
    title='Forecast Uncertainty by Product (% of historical mean)',
    labels={'MeanRelativeUncertainty': 'Relative Uncertainty (%)'}
)
fig.show()

In [6]:
risk_counts = inventory_risk['RiskStatus'].value_counts().reset_index()
fig2 = px.pie(
    risk_counts,
    names='RiskStatus',
    values='count',
    title='Inventory Risk Status Distribution',
    color='RiskStatus',
    color_discrete_map={
        'Stockout Risk': 'red',
        'Low Stock': 'orange',
        'Adequate': 'green',
        'Overstock Risk': 'purple',
        'Unreliable Forecast': 'gray'
    }
)
fig2.show()

print(inventory_risk[['StockCode', 'Description', 
                        'CurrentStock', 'ForecastedDemand',
                        'RiskStatus', 'RiskReason']].to_string(index=False))

StockCode                        Description  CurrentStock  ForecastedDemand          RiskStatus                                                     RiskReason
    23843        PAPER CRAFT , LITTLE BIRDIE   7864.004087      24689.088394 Unreliable Forecast Data too sparse/spike-driven for a trustworthy demand estimate
    22197                     POPCORN HOLDER   3384.579145       6739.630442           Low Stock                             May run short if demand is average
   85099B            JUMBO BAG RED RETROSPOT   3113.983688       4404.930377           Low Stock                             May run short if demand is average
    84077  WORLD WAR 2 GLIDERS ASSTD DESIGNS   3644.466401       4003.955492           Low Stock                             May run short if demand is average
    84879      ASSORTED COLOUR BIRD ORNAMENT   2425.559004       3700.143757           Low Stock                             May run short if demand is average
    23203           JUMBO BAG VINTAGE DO

# Day 3 — Notes & Observations

## Data Validation

**Data loading cell** — Loaded `uncertainty_metrics.csv` and `inventory_risk.csv`, confirming that both tables correctly carried the `DataQualityFlag` from earlier pipeline stages.

**Uncertainty bar chart** — Plotted `MeanRelativeUncertainty` for all products, colored by data-quality status. The two flagged products (23843: 1371.2%, 23166: 1056.7%) stand far above the 18 "OK" products, which cluster between roughly 210% and 460%. Although computed differently, both the uncertainty metric and the data-quality flag identify the same two products as unreliable forecasting candidates.

**Inventory risk pie chart** — Final risk distribution: 60% Low Stock, 30% Adequate, 10% Unreliable Forecast, and 0% Stockout Risk.

---

## Fix 1: Uncertainty metric hidden by clipping

`calculate_uncertainty_metrics` originally computed relative uncertainty as:

```
IntervalWidth / yhat
```

However, product 23166's forecast (`yhat`) was clipped to exactly zero because Prophet produced negative future predictions. This caused its relative uncertainty to appear as 0.0%, which is the opposite of reality.

The fix was to divide by each product's historical mean demand instead. Historical mean demand remains positive for every product in this dataset and correctly moved product 23166 to the second-highest uncertainty ranking (1056.7%).

---

## Fix 2: Inventory risk misclassifying unreliable products

The same clipping issue affected `calculate_inventory_risk`. Product 23166 was initially classified as "Adequate" because the forecasted demand was zero while synthetic stock remained positive.

This classification was technically correct given the clipped forecast, but misleading in practice because the product's demand is not truly zero—it is simply too irregular to forecast reliably.

The fix was to override the computed status:

- Products with `DataQualityFlag == 'OK'` receive a calculated inventory status.
- Products with `DataQualityFlag != 'OK'` are explicitly labeled **"Unreliable Forecast."**

---

## Fix 3: Stock-demand horizon mismatch

Initially, 80% of products were classified as "Low Stock." Part of this result came from a genuine bug: synthetic stock assumed a 3-week inventory horizon, while forecasted demand covered 4 weeks.

This mismatch structurally inflated the Low Stock count regardless of actual demand patterns.

After aligning both windows to four weeks, the distribution changed to:

- 60% Low Stock
- 30% Adequate
- 10% Unreliable Forecast
- 0% Stockout Risk

After the fix, product 23203 moved from "Stockout Risk" to "Low Stock" because the estimated stock assumption increased from three weeks to four weeks of average demand.

The remaining 60% Low Stock pattern appears genuine rather than buggy. Synthetic stock is based on historical average demand, while forecasted demand follows each product's fitted trend. Since many products show upward trends (see Day 2), a backward-looking average naturally underestimates future demand.

---

## Business Interpretation

### Why wide confidence intervals matter

A narrow confidence interval (for example, product 84946 at 210%) suggests that Prophet is relatively confident in its forecast, making inventory decisions based on the point estimate safer.

A wide confidence interval (for example, product 21915 at 460%, and especially products 23843 and 23166) means that true demand could reasonably fall across a much wider range.

Two products may have similar point forecasts but require very different inventory strategies if one forecast is substantially more uncertain than the other.

This is exactly why the **"Unreliable Forecast"** label matters: for products 23843 and 23166, uncertainty is so large that the point forecast alone is not useful for automated stocking decisions.

### Why many products still show "Low Stock"

Even after fixing the horizon mismatch, most products remain classified as Low Stock.

This is not necessarily evidence of a forecasting error. Current stock is estimated using historical averages, while the forecast extrapolates future trends. For products with growing demand, historical averages naturally lag behind future expectations, producing a large number of Low Stock alerts.

---

## Limitations

The inventory analysis still has one major limitation that cannot be fixed in code.

`EstimatedCurrentStock` is entirely synthetic because the UCI Online Retail dataset contains no real inventory information.

Aligning the stock and demand horizons removes a modeling inconsistency, but it does not make the underlying stock estimates realistic.

As a result:

- Inventory quantities should be interpreted as illustrative rather than operational.
- Risk categories depend heavily on the synthetic stock assumption.
- The system demonstrates how forecasts can be converted into business alerts, but it is not a production-ready inventory management system.

This limitation belongs in the README as a limitation of the dataset rather than a limitation of the forecasting model itself.